# conv-windowing-2d — worked example 3: 2-D max-pool via a non-overlapping window view

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-2d`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Max pooling reuses the same windowing idea but with a *between-window* step equal to the kernel size, so windows tile without overlap. Build the `(B, C, OH, OW, KH, KW)` view with middle strides `(s_h*KH, s_w*KW)`, then reduce the trailing `(KH, KW)` axes with `max` instead of contracting with a kernel.

## Worked solution

**Step 1 — output shape for non-overlapping pooling.** With stride = kernel size `K`, `OH = H // K` and `OW = W // K`. Each output cell covers a disjoint `KxK` tile.

**Step 2 — between-window stride is multiplied.** Unlike stride-1 conv, moving one step in `OH` must jump `K` input rows, so the middle pair is `(s_h * K, s_w * K)`. The trailing pair stays `(s_h, s_w)` to read every pixel inside the tile.

**Step 3 — build the view.** `as_strided(size=(B, C, OH, OW, K, K), stride=(s_b, s_c, s_h*K, s_w*K, s_h, s_w))`. No overlap means each storage element is visited once — still a view, no copy.

**Step 4 — reduce, don't contract.** Pooling has no learnable kernel: collapse the last two axes with `reduce(..., 'b c oh ow kh kw -> b c oh ow', 'max')`.

**Step 5 — verify against `F.max_pool2d(x, K)`.** They match because both take the max over identical disjoint tiles.

In [ ]:
import torch.nn.functional as F
from einops import reduce

def maxpool2d_via_windows(x: Tensor, K: int) -> Tensor:
    B, C, H, W = x.shape
    OH = H // K
    OW = W // K
    s_b, s_c, s_h, s_w = x.stride()
    windows = x.as_strided(
        size=(B, C, OH, OW, K, K),
        stride=(s_b, s_c, s_h * K, s_w * K, s_h, s_w),
    )
    return reduce(windows, 'b c oh ow kh kw -> b c oh ow', 'max')

t.manual_seed(0)
x = t.randn(2, 3, 8, 8)
out = maxpool2d_via_windows(x, K=2)
ref = F.max_pool2d(x, 2)
print(out.shape, bool(t.allclose(out, ref, atol=1e-5)))